# Week 14: Thermal IR Band 7: brightness temperature and hotspots

**Track:** Remote Sensing Specialist (Intermediate)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/14/](https://launchdetect.com/academy/week/14/)  
**Track index:** [https://launchdetect.com/academy/remote-sensing-specialist/](https://launchdetect.com/academy/remote-sensing-specialist/)

---

_The heart of LaunchDetect's methodology. Band 7 at 3.9 µm sees thermal emission strongly — rocket plumes show up as ~340 K hotspots against a background of ~290 K. This week you build a working hotspot detector._


## Why this week matters

This is THE LaunchDetect methodology week. The Planck inversion + threshold detection in this lab is the same logic running in production right now, scoring every NOAA GOES Band 7 frame against the spaceport registry. Master this and you can build your own launch-detection service from scratch.


## Learning objectives

By the end of this lab you will be able to:

- Convert raw ABI Band 7 radiance to brightness temperature in Kelvin
- Identify a rocket plume's thermal signature in Band 7
- Set a brightness-temperature threshold for hotspot detection
- Distinguish a real plume from a wildfire from a hot industrial source


## Setup — and why these dependencies

This lab uses: `leafmap[common] xarray rasterio netCDF4 satpy`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] xarray rasterio netCDF4 satpy


## The approach (and why)

Synthetic data demonstrates the Planck inversion + threshold flow without requiring a NetCDF download. Replace the synthetic radiance grid with a real one in extension work — the rest of the code is unchanged.


In [ ]:
# Week 14: thermal-IR Band 7 plume detection — convert radiance to brightness temperature.
import numpy as np

# Synthetic example — replace with real GOES-18 Band 7 NetCDF
# Constants from a typical Band 7 calibration:
fk1, fk2, bc1, bc2 = 2.0249e+05, 3.6989e+03, 0.43361, 0.99902

# Simulated radiance grid with a hot pixel (the plume) in the middle
rad = np.random.normal(80, 5, size=(20, 20))   # background
rad[9:11, 9:11] = 220                          # plume hotspot

# Planck inversion -> brightness temperature (Kelvin)
tb = (fk2 / np.log(fk1 / rad + 1) - bc1) / bc2

print(f"Background T_b mean: {tb[:5, :5].mean():.1f} K")
print(f"Plume   T_b max:    {tb.max():.1f} K")

# Threshold detection
plume_mask = tb > 320
print(f"Detected pixels above 320 K: {plume_mask.sum()}")

# Visualize
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(tb, cmap="inferno"); axes[0].set_title("Brightness temperature (K)")
axes[1].imshow(plume_mask, cmap="Reds"); axes[1].set_title("Hotspot mask (>320 K)")
plt.tight_layout(); plt.show()

# TODO: download a REAL GOES-18 Band 7 NetCDF for a known launch event and
# run the same conversion + threshold. Output (timestamp, lat, lon, T_b) records.


## What just happened — and why it works

Radiance is what the sensor measures. Brightness temperature is the physically meaningful quantity (temperature a perfect black body would need to emit that radiance). The threshold at 320 K is conservative; production uses 320-340 K depending on false-positive tolerance.


## Common gotchas

- Planck constants vary per band. Always read fk1/fk2/bc1/bc2 from the NetCDF, never hardcode.
- A hot pixel might be a plume, a wildfire, a gas flare, a sun-glint reflection, or industrial heat. Threshold alone is not enough — geofencing + temporal pattern + FIRMS overlap filtering eliminates ~95% of false positives.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/14/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
